# 05 · Selección del estudiante

El producto final corre **en dispositivos livianos** (teléfonos, placas tipo
Raspberry Pi) con TFLite, así que lo que se compara es el modelo que de verdad
se despliega: **cuantizado a int8**, no el `.keras` en float32.

Cuantizar después de entrenar cuesta precisión, y no lo mismo en cada
arquitectura: MobileNetV3 y EfficientNet usan hard-swish/swish y
squeeze-excite, que se cuantizan peor que las ReLU6 de MobileNetV2. Por eso se
mide el macro-F1 del modelo int8 y no se asume igual al de float32.

**Criterio: el mejor macro-F1 int8 en validación que quepa en `PRESUPUESTO_MB`.**
El presupuesto es una decisión de producto, no un óptimo.

Necesita el dataset adjunto: calibra con imágenes de train y evalúa sobre
validate y test.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path("/kaggle").exists():
    from kaggle_secrets import UserSecretsClient

    CLON = "/tmp/InsectModeling"
    AYUDA = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not Path(CLON).exists():
        subprocess.run(["git", "-c", f"credential.helper={AYUDA}", "clone",
                        "https://github.com/santiagoortizc/InsectModeling.git", CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import pandas as pd

from scripts import corpus, cuantizacion, evalStats
from scripts import modelRegistry as registro
from scripts.vizStyle import INK_MUTED, SERIE_1, SERIE_2, SERIE_3, applyStyle

PRESUPUESTO_MB = 8.0        # archivo .tflite int8, lo que se instala en el dispositivo
CORPUS = corpus.manifiesto()["creado"]
corpus.materializar()

# clave (arquitectura, descongelado): congelado y fine-tuning son regímenes
# distintos y ambos deben verse en la tabla, no solo el más reciente
runPorArquitectura = {}
for run in registro.listRuns():
    clave = (run.get("architecture"), run.get("finetuning", 0.0))
    if (run.get("role") == "estudiante" and run.get("dataset") == CORPUS
            and clave not in runPorArquitectura):
        runPorArquitectura[clave] = run

print(f"corpus {CORPUS[:10]} · {len(runPorArquitectura)} runs")

## 1 · Calidad y costo de despliegue

Cada run se convierte a `.tflite` con cuantización entera completa (pesos y
activaciones en int8, calibradas con 200 imágenes de train) y se evalúa con el
intérprete de TFLite. `int8_MB` es el tamaño real del archivo. `perdida_f1` es
lo que costó cuantizar en test.

La latencia es con un hilo. `motor` dice si corrió con XNNPACK o con los kernels
de referencia (XNNPACK no acepta algunos modelos int8): entre motores distintos
las latencias no se comparan.

In [ ]:
filas = []
for (arquitectura, descongelado), run in runPorArquitectura.items():
    runId = run["run_id"]
    clases = registro.loadRunMetrics(runId)["class_names"]
    ruta = cuantizacion.convertirRun(runId)
    test = cuantizacion.evaluar(ruta, "dataset/test", clases)
    validate = cuantizacion.evaluar(ruta, "dataset/validate", clases)
    latencia, desviacion, motor = cuantizacion.medirLatencia(ruta)
    filas.append({
        "arquitectura": arquitectura,
        "descongelado": descongelado,
        "validate_macro_f1": validate["macro_f1"],
        "test_macro_f1": test["macro_f1"],
        "test_macro_f1_f32": run["test_macro_f1"],
        "perdida_f1": run["test_macro_f1"] - test["macro_f1"],
        "test_accuracy": test["accuracy"],
        "top3": test["top3_accuracy"],
        "int8_MB": round(cuantizacion.tamanoMB(ruta), 2),
        "latencia_ms": round(latencia, 1),
        "latencia_sd_ms": round(desviacion, 1),
        "motor": motor,
        "run_id": runId,
    })
    print(f"{arquitectura}: macro-F1 int8 {test['macro_f1']:.4f} "
          f"(float32 {run['test_macro_f1']:.4f}) · {filas[-1]['int8_MB']} MB")

comparacion = pd.DataFrame(filas).sort_values("validate_macro_f1", ascending=False).reset_index(drop=True)
comparacion.drop(columns="run_id")

## 2 · Calidad vs tamaño

Cada flecha va del macro-F1 en float32 al int8: su largo es lo que costó
cuantizar.

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

dentro = comparacion["int8_MB"] <= PRESUPUESTO_MB

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(0, PRESUPUESTO_MB, color=SERIE_3, alpha=0.10)
ax.axvline(PRESUPUESTO_MB, color=INK_MUTED, linestyle="--", linewidth=1.2)
ax.text(PRESUPUESTO_MB, comparacion["test_macro_f1"].min(), f" presupuesto {PRESUPUESTO_MB:.0f} MB",
        fontsize=9, va="bottom", color=INK_MUTED)
for _, f in comparacion.iterrows():
    ax.annotate("", xy=(f["int8_MB"], f["test_macro_f1"]),
                xytext=(f["int8_MB"], f["test_macro_f1_f32"]),
                arrowprops={"arrowstyle": "->", "color": INK_MUTED})
ax.scatter(comparacion.loc[dentro, "int8_MB"], comparacion.loc[dentro, "test_macro_f1"],
           s=90, color=SERIE_1, label="cabe", zorder=3)
ax.scatter(comparacion.loc[~dentro, "int8_MB"], comparacion.loc[~dentro, "test_macro_f1"],
           s=90, color=SERIE_2, label="excede", zorder=3)
for _, f in comparacion.iterrows():
    ax.annotate(f["arquitectura"], (f["int8_MB"], f["test_macro_f1"]),
                textcoords="offset points", xytext=(8, 5), fontsize=8)
ax.set(xlabel="Tamaño del modelo int8 (MB)", ylabel="macro F1 int8 (test)",
       title="Candidatos a estudiante · calidad vs tamaño")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 3 · Estudiante elegido

In [ ]:
candidatos = comparacion[comparacion["int8_MB"] <= PRESUPUESTO_MB]
if candidatos.empty:
    raise ValueError(f"Ningún candidato cabe en {PRESUPUESTO_MB} MB")

elegido = candidatos.sort_values("validate_macro_f1", ascending=False).iloc[0]
run = runPorArquitectura[(elegido["arquitectura"], elegido["descongelado"])]

registro.writeJson(registro.OUTPUTS_DIR / "student.json", {
    "run_id": run["run_id"],
    "architecture": elegido["arquitectura"],
    "finetuning": float(elegido["descongelado"]),
    "dataset": CORPUS,
    "formato": "tflite int8",
    "test_accuracy": float(elegido["test_accuracy"]),
    "test_macro_f1": float(elegido["test_macro_f1"]),
    "test_macro_f1_f32": float(elegido["test_macro_f1_f32"]),
    "validate_macro_f1": float(elegido["validate_macro_f1"]),
    "top3_accuracy": float(elegido["top3"]),
    "pesos_MB": float(elegido["int8_MB"]),
    "latencia_ms": float(elegido["latencia_ms"]),
    "latencia_sd_ms": float(elegido["latencia_sd_ms"]),
    "motor": elegido["motor"],
    "presupuesto_MB": PRESUPUESTO_MB,
})

print(f"Estudiante: {elegido['arquitectura']} (descongelado {elegido['descongelado']}) · "
      f"macro-F1 int8 {elegido['test_macro_f1']:.4f} (float32 {elegido['test_macro_f1_f32']:.4f}) · "
      f"top-3 {elegido['top3']:.4f} · {elegido['int8_MB']:.1f} MB")

for _, f in comparacion.iterrows():
    if f["arquitectura"] != elegido["arquitectura"] and f["test_macro_f1_f32"] > elegido["test_macro_f1_f32"]:
        print(f"  {f['arquitectura']} era mejor en float32 ({f['test_macro_f1_f32']:.4f}) "
              f"pero cuantizado queda en {f['test_macro_f1']:.4f}"
              + ("" if f["int8_MB"] <= PRESUPUESTO_MB else f" y pesa {f['int8_MB']:.1f} MB"))

from scripts import pesos
from scripts.kaggle import publicarArchivos
if Path("/kaggle").exists():
    pesos.subir(run["run_id"], registro.runDir(run["run_id"]) / cuantizacion.ARCHIVO,
                cuantizacion.ARCHIVO)
    publicarArchivos(f"Select {elegido['arquitectura']} as student",
                     "outputs/student.json")

## Resumen

In [ ]:
comparacion.drop(columns='run_id')